# Estud-IA · IA nivel intermedio · Clase 5
**Decidir si usar el modelo tal cual, entrenar una salida o afinar una parte, y diseñar el experimento**

Ejecuta las celdas en orden (Ctrl + Enter). Ocho partes:
0. Entorno
1. Los mismos datos y la misma partición de la clase 4
2. Tres caminos para usar un modelo que ya existe
3. Camino A · usar el modelo tal cual (y cambiar una sola cosa)
4. Camino A con ejemplos en el prompt (chat de IA generativa)
5. Camino B · representaciones congeladas y una salida entrenada por ti
6. Camino C · afinar parte del modelo: se diseña, no se entrena
7. Comparar y mirar qué errores cambian
8. Tu ficha de hipótesis

**Hilo de la clase:** en la clase 4 medimos un modelo y separamos sus errores. Hoy decidimos qué cambio probar primero y lo escribimos como un experimento que otra persona pueda repetir.

**Cuida la información:** los mensajes son ficticios. Si usas datos propios, que sean con permiso y sin datos personales.

## Parte 0 · Prepara el entorno
`pandas` y `scikit-learn` leen la tabla y entrenan la salida pequeña. `torch` y `transformers` cargan el modelo que convierte frases en números. Fijamos la misma semilla de la clase 4 para tener la misma partición.

In [ ]:
import os, re, io, json, time
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModel, pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import GroupShuffleSplit

SEMILLA = 1540
CLASES = ["facturación", "envío", "devolución"]
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

# Cuaderno de experimentos: cada prueba queda escrita con su único cambio
LOG = []
def registrar(experimento, cambio, reales, predichas):
    k = int((np.array(reales) == np.array(predichas)).sum())
    LOG.append({"experimento": experimento, "único cambio": cambio, "aciertos": f"{k} de {len(reales)}"})
    return k

print("transformers", transformers.__version__, "| torch", torch.__version__)

## Parte 1 · Los mismos datos y la misma partición
Es el archivo de la clase 4: 344 mensajes ficticios de una tienda en línea, tres clases, dos revisores y un `grupo` para los mensajes casi repetidos. Esta celda lo guarda en tu Colab.

In [ ]:
CSV_TEXTO = r'''id,fecha,canal,mensaje,etiqueta,etiqueta_revisor2,ambiguo,grupo
1,2026-09-02,correo,"parce, el descuento del cupón no se aplicó al compra tm-45709, pagué de más",facturación,facturación,0,G055
2,2026-09-02,whatsapp,"Que mas, no quiero devolver nada, solo saber donde esta mi compra TM-41379",envío,envío,0,G082
3,2026-09-02,whatsapp,"Buenos días, el pedido salió para Cali pero yo vivo en otro barrio, se va a devolver a bodega?",envío,envío,0,G122
4,2026-09-02,whatsapp,"Buenas, necesito que le cambien el nombre a la factura, salió con el de mi mamá. es urgente",facturación,facturación,0,G133
5,2026-09-02,correo,"Hola, buenas, pille, devolver los tenis me sale más caro que el producto? de qué forma funciona eso",devolución,devolución,0,G262
6,2026-09-03,whatsapp,"insisto, no quiero la camiseta, me arrepentí de la compra. se puede devolver?",devolución,devolución,0,G006
7,2026-09-03,whatsapp,"Hola, buenas, la plancha no calienta parejo, me parece un defecto de fábrica, deseo cambio",devolución,devolución,0,G126
8,2026-09-03,whatsapp,xq me cobraron el domicilio si la promo era envío gratis? pedido TM-41024,facturación,facturación,0,G148
9,2026-09-03,web,"Cordial saludo, no quiero la licuadora, me arrepentí de la compra. se puede devolver?",devolución,devolución,0,G149
10,2026-09-03,whatsapp,"no quiero la camiseta, me arrepentí de la compra. se puede devolver?",devolución,devolución,0,G006
11,2026-09-03,web,"Buenos días, me cobraron dos veces la chaqueta con el compra TM-49706, necesito que me reversen uno",facturación,facturación,0,G221
12,2026-09-03,correo,"otra vez: no quiero la camiseta, me arrepentí de la compra. se puede devolver?",devolución,devolución,0,G006
13,2026-09-04,correo,hice el cambio por talla pero el nuevo pedido no ha salido de la bodega,envío,devolución,1,G040
14,2026-09-04,web,"Buenas, el paquete está detenido en una bodega de Pereira desde hace 10 días, pueden escalarlo? ... es urgente",envío,envío,0,G081
15,2026-09-04,correo,"el repartidor llamó pero no contesté, cómo reprogramo la entrega del TM-48132?",envío,envío,0,G111
16,2026-09-04,whatsapp,"Parce, el paquete llegó a la sede de Envigado y nadie me avisó, hace 8 días",envío,envío,0,G021
17,2026-09-04,correo,"reitero: parce, el paquete llegó a la sede de Envigado y nadie me avisó, hace 8 días",envío,envío,0,G021
18,2026-09-04,whatsapp,"la guía de devolución que me enviaron no sirve, sale error al imprimirla",devolución,envío,1,G181
19,2026-09-04,whatsapp,"Buenos días, llevo 6 días esperando la plancha y el seguimiento no avanza desde Cali",envío,envío,0,G192
20,2026-09-04,whatsapp,"Buenas, quiero devolver el morral, no es lo que esperaba. pedido TM-45185",devolución,devolución,0,G199
21,2026-09-05,whatsapp,"sigo esperando respuesta: hola, mi vecina recibió el paquete pero la guía dice que lo recibí yo, quién firmó? ... un saludo",envío,envío,0,G010
22,2026-09-05,whatsapp,"Buenas, buenas, cuándo llega mi pedido a Cali? me urge para el fin de semana",envío,envío,0,G080
23,2026-09-05,correo,"Hola, mi vecina recibió el paquete pero la guía dice que lo recibí yo, quién firmó? ... un saludo",envío,envío,0,G010
24,2026-09-05,web,"cordial saludo, me dijeron que llegaba en 6 días y ya pasó el plazo, qué pasó con la chaqueta?",envío,envío,0,G157
25,2026-09-05,whatsapp,"Que mas, llevo 10 dias esperando la camiseta y el seguimiento no avanza desde Medellin",envío,envío,0,G219
26,2026-09-06,correo,"insisto, buenas, buenas, cuando llega mi pedido a Barranquilla? me urge para el fin de semana",envío,envío,0,G025
27,2026-09-06,web,"Hola, requiero la factura electrónica dla orden TM-44483, no me llegó al correo",facturación,facturación,0,G096
28,2026-09-06,whatsapp,"buenas tardes, la factura llegó en dólares y yo pagué en pesos, eso de qué forma es?",facturación,facturación,0,G125
29,2026-09-06,whatsapp,"reitero: hola, mi vecina recibió el paquete pero la guía dice que lo recibí yo, quién firmó? ... un saludo",envío,envío,0,G010
30,2026-09-06,correo,"Qué más, compré el cargador para regalo y no le gustó, aún está sin usar, lo puedo devolver?",devolución,devolución,0,G140
31,2026-09-06,correo,"Buenas, buenas, cuando llega mi pedido a Barranquilla? me urge para el fin de semana",envío,envío,0,G025
32,2026-09-06,whatsapp,"Buenas tardes, en que horario pasan a entregar en Envigado? trabajo hasta las 6",envío,envío,0,G012
33,2026-09-06,whatsapp,"Buenas tardes, el paquete llegó a la sede de Medellín y nadie me avisó, hace 3 días",envío,envío,0,G210
34,2026-09-07,whatsapp,"Buenos días, puedo recoger el pedido en una oficina? no hay quien reciba en la casa",envío,envío,0,G014
35,2026-09-07,correo,"qué más, el morral llegó defectuoso, no enciende. requiero que me devuelvan la plata",devolución,devolución,0,G092
36,2026-09-07,web,"Qué más, en la página decía $79.000 y en el extracto me cobraron $39.900, a qué se debe la diferencia? ... gracias",facturación,facturación,0,G109
37,2026-09-07,correo,quiero saber si el reembolso me llega antes o después de que recojan el paquete,devolución,envío,1,G256
38,2026-09-07,correo,"insisto, buenas tardes, en que horario pasan a entregar en Envigado? trabajo hasta las 6",envío,envío,0,G012
39,2026-09-07,whatsapp,"Parce, cuántos días tengo para devolver la camiseta? lo recibí desde hace 6",devolución,devolución,0,G297
40,2026-09-08,correo,"sigo esperando respuesta: buenas tardes, en que horario pasan a entregar en Envigado? trabajo hasta las 6",envío,envío,0,G012
41,2026-09-08,correo,"Hola, requiero una nota crédito del compra TM-45278, contabilidad me la está pidiendo",facturación,facturación,0,G005
42,2026-09-08,correo,"Que mas, ya no necesito la licuadora, lo compre doble por error, de que forma lo devuelvo?",devolución,devolución,0,G114
43,2026-09-08,whatsapp,"Parce, pagué con PSE y el banco dice aprobado pero ustedes no registran nada, pedido TM-43162, mil gracias",facturación,facturación,0,G117
44,2026-09-08,correo,"Cordial saludo, llegó la caja pero sin el cargador, solo el empaque, qué pasó en el despacho?",envío,envío,0,G158
45,2026-09-08,correo,"Parce, el repartidor llamó pero no contesté, de qué forma reprogramo la entrega del TM-48934?, quedo atento",envío,envío,0,G171
46,2026-09-08,correo,"Parce, no me cobren otra vez el pedido TM-46013, ya lo pagué y tengo el comprobante",facturación,facturación,0,G173
47,2026-09-08,whatsapp,"Qué más, el valor que me cobraron no coincide con la cotización que me enviaron, es $79.500 vs $119.900, es urgente",facturación,facturación,0,G209
48,2026-09-08,whatsapp,"Cordial saludo, dicen que mañana llega y llevo tres mañanas esperando, esto es normal?",envío,envío,0,G004
49,2026-09-08,web,"Cordial saludo, el cargador vino con una pieza rota y sin manual, lo deseo devolver, un saludo",devolución,devolución,0,G296
50,2026-09-09,whatsapp,"insisto, cordial saludo, dicen que mañana llega y llevo tres mañanas esperando, esto es normal?",envío,envío,0,G004
51,2026-09-09,whatsapp,"Hola, buenas, redeseo la guía para devolver el termo, ustedes la pagan o yo?",devolución,devolución,0,G152
52,2026-09-09,correo,"insisto, buenos días, puedo recoger el pedido en una oficina? no hay quien reciba en la casa",envío,envío,0,G014
53,2026-09-09,whatsapp,"hola, buenas, dónde está mi pedido? el seguimiento dice 'en reparto' desde ayer, mil gracias",envío,envío,0,G180
54,2026-09-09,correo,"Parce, me llego incompleto: pedi 2 cosas y solo vino 1, y la otra? pedido TM-42920",envío,envío,0,G226
55,2026-09-09,whatsapp,"de nuevo: hola, requiero una nota crédito del compra TM-45278, contabilidad me la está pidiendo",facturación,facturación,0,G005
56,2026-09-09,whatsapp,"Parce, me descontaron la plata de la tarjeta pero la plataforma dice que el pago fue rechazado (TM-43430)",facturación,facturación,0,G259
57,2026-09-10,whatsapp,"Cordial saludo, devolví la camiseta y me dijeron que llegó en mal estado, pero salió perfecto de mi casa ... gracias",devolución,devolución,0,G116
58,2026-09-10,correo,"llegó la caja pero sin el bolso, solo el empaque, qué pasó en el despacho?",envío,envío,0,G166
59,2026-09-11,correo,"buenas, no me sirve lo que me enviaron, qué pasos sigo para devolverlo?",devolución,devolución,0,G230
60,2026-09-12,correo,"buenas, el cupón de bienvenida me lo restaron en un compra pero en el otro no, por qué?",facturación,facturación,0,G038
61,2026-09-12,whatsapp,"requiero que le cambien el nombre a la factura, salió con el de mi mamá",facturación,facturación,0,G105
62,2026-09-12,web,"Cordial saludo, por favor confirmar que recibieron el pago de $149.900 que hice ayer por Nequi. mil gracias",facturación,facturación,0,G119
63,2026-09-12,web,"Cordial saludo, solicito cambio por garantía de el bolso, pedido TM-45201, compré en Manizales. mil gracias",devolución,devolución,0,G277
64,2026-09-13,whatsapp,"Cordial saludo, la orden salio para Envigado pero yo vivo en otro barrio, se va a devolver a bodega?",envío,envío,0,G110
65,2026-09-13,correo,"insisto, buenas tardes, pille, devolver la camiseta me sale más caro que el producto? cómo funciona eso",devolución,devolución,0,G034
66,2026-09-13,correo,"Buenas, la factura del compra TM-49620 salió con otro NIT, la necesito a nombre de mi empresa. les agradezco",facturación,facturación,0,G223
67,2026-09-13,whatsapp,"necesito la factura electrónica dla orden TM-41478, no me llegó al correo, mil gracias",facturación,facturación,0,G224
68,2026-09-13,correo,el mensajero dejo el pedido con un desconocido y yo no autorice eso,envío,envío,0,G232
69,2026-09-14,correo,"Buenas, me dijeron que llegaba en 5 días y ya pasó el plazo, qué pasó con los tenis?",envío,envío,0,G091
70,2026-09-14,web,"Buenas tardes, pille, devolver la camiseta me sale más caro que el producto? cómo funciona eso",devolución,devolución,0,G034
71,2026-09-14,correo,"Parce, me equivoqué de talla en el pedido TM-43373, hay cambio gratis?",devolución,devolución,0,G003
72,2026-09-14,whatsapp,"me llegó dañado y la guía ya figura como recibido conforme, cómo reclamo?",devolución,envío,1,G196
73,2026-09-14,whatsapp,"Qué más, pille, se me atrasó el compra para el cumpleaños, hay forma de acelerarlo?",envío,envío,0,G234
74,2026-09-15,whatsapp,"insisto, buenas, me llegó incompleto: pedí 2 cosas y solo vino 1, y la otra? pedido TM-48632",envío,envío,0,G029
75,2026-09-15,whatsapp,"Cordial saludo, no me sirve lo que me enviaron, qué pasos sigo para devolverlo?",devolución,devolución,0,G115
76,2026-09-15,whatsapp,"buenas tardes, estoy dentro de los 5 días de retracto y quiero devolver el pedido tm-42811",devolución,devolución,0,G191
77,2026-09-16,whatsapp,"Buenas tardes, el termo vino con una falla en la costura, quiero que lo recojan",devolución,devolución,0,G071
78,2026-09-16,correo,"insisto, parce, me equivoqué de talla en el pedido TM-43373, hay cambio gratis?",devolución,devolución,0,G003
79,2026-09-16,web,"Buenas, en qué horario pasan a entregar en Bogotá? trabajo hasta las 6",envío,envío,0,G129
80,2026-09-16,correo,"otra vez: parce, me equivoqué de talla en el pedido TM-43373, hay cambio gratis?",devolución,devolución,0,G003
81,2026-09-16,whatsapp,"Buenas tardes, buenas, el cupón de bienvenida me lo restaron en un pedido pero en el otro no, por qué?, mil gracias",facturación,facturación,0,G163
82,2026-09-16,whatsapp,"Qué más, por qué me cobraron IVA si el producto aparecía sin IVA?",facturación,facturación,0,G263
83,2026-09-16,correo,"Hola, buenas, el paquete esta detenido en una bodega de Manizales hace 3 dias, pueden escalarlo?. un saludo",envío,envío,0,G264
84,2026-09-16,correo,"hola, mi pedido tm-44818 sigue sin llegar, ya van 10 dias y el seguimiento no se mueve",envío,envío,0,G292
85,2026-09-17,web,"Buenas, me llegó incompleto: pedí 2 cosas y solo vino 1, y la otra? pedido TM-48632",envío,envío,0,G029
86,2026-09-17,whatsapp,"Hola, buenas, no me cobren otra vez el pedido TM-40243, ya lo pagué y tengo el comprobante, mil gracias",facturación,facturación,0,G104
87,2026-09-17,web,"Buenas tardes, necesito una nota credito del pedido TM-41098, contabilidad me la esta pidiendo, les agradezco",facturación,facturación,0,G134
88,2026-09-17,whatsapp,"Buenos días, llevo 6 días esperando los audífonos y el seguimiento no avanza desde Bogotá",envío,envío,0,G239
89,2026-09-18,correo,"cordial saludo, el paquete llegó a la sede de manizales y nadie me avisó, desde hace 5 días",envío,envío,0,G026
90,2026-09-18,web,"sigo esperando respuesta: cordial saludo, el paquete llegó a la sede de manizales y nadie me avisó, desde hace 5 días",envío,envío,0,G026
91,2026-09-18,correo,"Buenas, quiero el certificado de retención del año, somos clientes frecuentes",facturación,facturación,0,G138
92,2026-09-18,web,"Parce, mi vecina recibió el paquete pero el seguimiento dice que lo recibí yo, quién firmó? ... les agradezco",envío,envío,0,G186
93,2026-09-18,correo,"hola, buenas, no me gustó el color, puedo cambiarlo por otro? pedido tm-44911 ... es urgente",devolución,devolución,0,G233
94,2026-09-19,correo,"Hola, buenas, cómo es el proceso de garantía? el termo dejó de funcionar a las 2 semanas",devolución,devolución,0,G074
95,2026-09-19,whatsapp,"mi tarjeta quedó con dos cargos iguales del TM-47403, deseo saber cuál es el bueno",facturación,facturación,0,G184
96,2026-09-20,whatsapp,"buenas tardes, el domiciliario no encuentra la dirección, está en Medellín, me pueden llamar?",envío,envío,0,G098
97,2026-09-20,correo,"Buenas tardes, no quiero devolver nada, solo saber dónde está mi compra TM-43907 ... es urgente",envío,envío,0,G015
98,2026-09-20,correo,"Buenas, solicito cambio por garantía de el morral, compra TM-47190, compré en Rionegro",devolución,devolución,0,G252
99,2026-09-20,whatsapp,"insisto, buenas tardes, no quiero devolver nada, solo saber dónde está mi compra TM-43907 ... es urgente",envío,envío,0,G015
100,2026-09-21,whatsapp,no me llegó la nota crédito de la devolución que hice en octubre,facturación,devolución,1,G131
101,2026-09-21,whatsapp,"Buenas, quiero el certificado de retención del año, somos clientes frecuentes. gracias",facturación,facturación,0,G195
102,2026-09-21,whatsapp,"no entiendo el cargo de $39.900 que aparece en mi cuenta, yo solo compré el reloj",facturación,facturación,0,G272
103,2026-09-21,whatsapp,"Hola, pagué la cuota inicial y me siguen enviando recordatorios de cobro, ya estoy al día",facturación,facturación,0,G287
104,2026-09-22,correo,"Parce, el chunche que pedí no ha llegado, el seguimiento TM-43867 no tiene novedades",envío,envío,0,G035
105,2026-09-22,correo,"de nuevo: cordial saludo, mi tarjeta quedó con dos cargos iguales del tm-44308, quiero saber cuál es el bueno, les agradezco",facturación,facturación,0,G023
106,2026-09-22,correo,"Hola, no me sirve lo que me enviaron, que pasos sigo para devolverlo?. les agradezco",devolución,devolución,0,G177
107,2026-09-22,whatsapp,"Hola, el repartidor llamó pero no contesté, cómo reprogramo la entrega del TM-40283?",envío,envío,0,G237
108,2026-09-23,whatsapp,"Parce, no me llego el codigo de seguimiento, como hago para rastrear la licuadora?",envío,envío,0,G036
109,2026-09-23,whatsapp,"cordial saludo, mi tarjeta quedó con dos cargos iguales del tm-44308, quiero saber cuál es el bueno, les agradezco",facturación,facturación,0,G023
110,2026-09-23,whatsapp,"Parce, pille, me aparece un cargo pendiente de $68.500 y no sé de qué es",facturación,facturación,0,G202
111,2026-09-23,whatsapp,"Buenas tardes, envié la orden de vuelta hace 12 días, cuándo me reembolsan?",devolución,devolución,0,G295
112,2026-09-24,correo,"Parce, quisiera saber el estado del despacho, la página me tiene en 'procesando' desde hace 5 días. gracias",envío,envío,0,G103
113,2026-09-24,whatsapp,"de nuevo: buenas, la transportadora dejó el paquete en la portería equivocada, compra TM-46633",envío,envío,0,G018
114,2026-09-24,whatsapp,"Qué más, la factura dla orden TM-47406 salió con otro NIT, la necesito a nombre de mi empresa",facturación,facturación,0,G281
115,2026-09-25,correo,"Qué más, dónde está mi compra? la guía dice 'en reparto' desde ayer",envío,envío,0,G112
116,2026-09-25,whatsapp,"Qué más, me cobraron una comisión por pagar contra entrega, nadie me avisó",facturación,facturación,0,G139
117,2026-09-25,web,"Buenos días, mi tarjeta quedó con dos cargos iguales del TM-47336, deseo saber cuál es el bueno",facturación,facturación,0,G245
118,2026-09-25,web,"Buenos días, el paquete llegó a la sede de Itagüí y nadie me avisó, desde hace 12 días",envío,envío,0,G253
119,2026-09-26,web,"Buenas, la transportadora dejó el paquete en la portería equivocada, compra TM-46633",envío,envío,0,G018
120,2026-09-26,whatsapp,"Buenos días, el chunche que pedí no ha llegado, la guía TM-42880 no tiene novedades",envío,envío,0,G106
121,2026-09-26,correo,"hola, me cobraron dos veces la chaqueta con la orden tm-45694, necesito que me reversen uno",facturación,facturación,0,G198
122,2026-09-26,whatsapp,"Cordial saludo, ñapa, me pueden decir si ya salio el compra? no me llego ni correo ni nada",envío,envío,0,G213
123,2026-09-27,whatsapp,"mi vecina recibió el paquete pero la guía dice que lo recibí yo, quién firmó?",envío,envío,0,G054
124,2026-09-27,whatsapp,"Hola, el seguimiento figura entregada pero yo no recibi nada, pedido TM-48820",envío,envío,0,G095
125,2026-09-28,correo,"Hola, puedo recoger el pedido en una oficina? no hay quien reciba en la casa",envío,envío,0,G176
126,2026-09-28,correo,"cambié la dirección porque la factura sale a otra ciudad, me pueden ayudar?",envío,facturación,1,G225
127,2026-09-28,correo,"me mandaron un producto usado, tiene rayones, quiero cambio por uno nuevo. quedo atento",devolución,devolución,0,G248
128,2026-09-29,web,"ya pasaron 5 días hábiles y el pedido sigue en 'preparando', se perdió o qué?",envío,envío,0,G145
129,2026-09-29,web,"Cordial saludo, el chunche que pedí no ha llegado, la guía TM-49276 no tiene novedades",envío,envío,0,G207
130,2026-09-30,whatsapp,"Buenos dias, puedo recoger el pedido en una oficina? no hay quien reciba en la casa",envío,envío,0,G062
131,2026-09-30,whatsapp,"Parce, me cobraron una comision por pagar contra entrega, nadie me aviso",facturación,facturación,0,G072
132,2026-09-30,correo,"Qué más, no quiero devolver nada, solo saber dónde está mi compra TM-43428",envío,envío,0,G211
133,2026-10-01,whatsapp,"Hola, buenas, quisiera saber el estado del despacho, la página me tiene en 'procesando' desde hace 8 días",envío,envío,0,G063
134,2026-10-01,whatsapp,"Cordial saludo, de qué forma es el proceso de garantía? la licuadora dejó de funcionar a las 2 semanas",devolución,devolución,0,G194
135,2026-10-01,whatsapp,"Buenas tardes, el descuento del cupón no se aplicó al compra TM-40376, pagué de más ... mil gracias",facturación,facturación,0,G200
136,2026-10-01,correo,"Buenas tardes, se me pasó el tiempo de retracto por un día, igual me pueden recibir el cargador?",devolución,devolución,0,G227
137,2026-10-01,whatsapp,"Hola, buenas, me dijeron que llegaba en 6 días y ya pasó el plazo, qué pasó con la lámpara?",envío,envío,0,G258
138,2026-10-01,web,"Buenas tardes, me avisaron que el pedido TM-47633 salió ayer, pero el link de seguimiento no abre ... mil gracias",envío,envío,0,G278
139,2026-10-01,web,"otra vez: buenos días, dónde está mi compra? la guía dice 'en reparto' desde ayer",envío,envío,0,G020
140,2026-10-02,web,"buenos días, dónde está mi compra? la guía dice 'en reparto' desde ayer",envío,envío,0,G020
141,2026-10-02,whatsapp,"reitero: buenos días, dónde está mi compra? la guía dice 'en reparto' desde ayer",envío,envío,0,G020
142,2026-10-02,whatsapp,"qué más, la guía figura entregada pero yo no recibí nada, pedido tm-48288, mil gracias",envío,envío,0,G179
143,2026-10-02,correo,"buenas tardes, por les pido el favor de confirmar que recibieron el pago de $119.900 que hice ayer por nequi, quedo atento",facturación,facturación,0,G189
144,2026-10-02,web,"Qué más, la factura del compra TM-41339 salió con otro NIT, la necesito a nombre de mi empresa ... quedo atento",facturación,facturación,0,G244
145,2026-10-02,whatsapp,"Hola, buenas, el producto no corresponde a la foto de la página, me siento engañado, lo devuelvo ... gracias",devolución,devolución,0,G275
146,2026-10-03,whatsapp,"Hola, buenas, buenas, cuándo llega mi compra a Barranquilla? me urge para el fin de semana",envío,envío,0,G009
147,2026-10-03,whatsapp,"Buenas tardes, la factura trae mal la cedula, favor corregirla, compra TM-46590",facturación,facturación,0,G243
148,2026-10-03,correo,"sigo esperando respuesta: buenos días, dónde está mi compra? la guía dice 'en reparto' desde ayer",envío,envío,0,G020
149,2026-10-03,correo,"Hola, la factura trae mal la cédula, favor corregirla, pedido TM-40212",facturación,facturación,0,G288
150,2026-10-04,whatsapp,"Parce, el mensajero dice que fue a mi casa y no había nadie, pero yo estuve todo el día",envío,envío,0,G156
151,2026-10-04,whatsapp,"parce, la licuadora huele a usado, parece reempacado, no lo quiero",devolución,devolución,0,G162
152,2026-10-04,web,"Hola, no me cobren otra vez la orden TM-43385, ya lo pagué y tengo el comprobante",facturación,facturación,0,G168
153,2026-10-04,web,"insisto, hola, buenas, buenas, cuándo llega mi compra a Barranquilla? me urge para el fin de semana",envío,envío,0,G009
154,2026-10-04,correo,"Qué más, el chunche que pedí no ha llegado, la guía TM-47704 no tiene novedades",envío,envío,0,G249
155,2026-10-05,correo,"Parce, no me gustó el color, puedo cambiarlo por otro? pedido TM-48655",devolución,devolución,0,G057
156,2026-10-05,whatsapp,"Hola, redeseo que entreguen en horario de oficina, en la tarde no hay nadie",envío,envío,0,G128
157,2026-10-05,web,"cordial saludo, solicito copia de la factura de la plancha para la garantía del fabricante ... quedo atento",facturación,facturación,0,G182
158,2026-10-05,web,"sigo esperando respuesta: hola, buenas, buenas, cuándo llega mi compra a Barranquilla? me urge para el fin de semana",envío,envío,0,G009
159,2026-10-05,correo,"Cordial saludo, pille, se me atrasó el compra para el cumpleaños, hay forma de acelerarlo?",envío,envío,0,G001
160,2026-10-06,whatsapp,"Buenas, buenas tardes, el domiciliario no encuentra la dirección, está en Manizales, me pueden llamar?",envío,envío,0,G083
161,2026-10-06,correo,"Buenas tardes, no entiendo el cargo de $189.000 que aparece en mi cuenta, yo solo compré la lámpara. les agradezco",facturación,facturación,0,G172
162,2026-10-06,web,"de nuevo: cordial saludo, pille, se me atrasó el compra para el cumpleaños, hay forma de acelerarlo?",envío,envío,0,G001
163,2026-10-06,whatsapp,"pagué la cuota inicial y me siguen enviando recordatorios de cobro, ya estoy al día",facturación,facturación,0,G019
164,2026-10-06,web,"reitero: pagué la cuota inicial y me siguen enviando recordatorios de cobro, ya estoy al día",facturación,facturación,0,G019
165,2026-10-06,web,"Qué más, el valor que me cobraron no coincide con la cotización que me enviaron, es $149.000 vs $89.000",facturación,facturación,0,G283
166,2026-10-06,whatsapp,"de nuevo: pagué la cuota inicial y me siguen enviando recordatorios de cobro, ya estoy al día",facturación,facturación,0,G019
167,2026-10-07,whatsapp,"Parce, en la página decía $89.900 y en el extracto me cobraron $79.000, a qué se debe la diferencia?",facturación,facturación,0,G185
168,2026-10-08,whatsapp,"Qué más, pille, devolver el cargador me sale más caro que el producto? cómo funciona eso",devolución,devolución,0,G060
169,2026-10-08,web,"Buenas, el repartidor llamó pero no contesté, cómo reprogramo la entrega del TM-49622?",envío,envío,0,G155
170,2026-10-08,whatsapp,el mensajero me quiere cobrar por recoger la devolución,devolución,envío,1,G220
171,2026-10-08,correo,"el compra salió para Bello pero yo vivo en otro barrio, se va a devolver a bodega?",envío,envío,0,G267
172,2026-10-08,correo,"Cordial saludo, porfa confirmar que recibieron el pago de $89.900 que hice ayer por Nequi. un saludo",facturación,facturación,0,G291
173,2026-10-08,whatsapp,"Cordial saludo, solicito copia de la factura de la plancha para la garantía del fabricante",facturación,facturación,0,G293
174,2026-10-09,whatsapp,"Qué más, puedo recoger el pedido en una oficina? no hay quien reciba en la casa",envío,envío,0,G136
175,2026-10-09,whatsapp,"Buenas tardes, me avisaron que el pedido TM-49341 salió ayer, pero el link de seguimiento no abre",envío,envío,0,G137
176,2026-10-10,whatsapp,"Hola, deseo que me recojan el bolso en la casa, no puedo ir a ninguna oficina. es urgente",devolución,devolución,0,G100
177,2026-10-11,whatsapp,"Buenas tardes, necesito que entreguen en horario de oficina, en la tarde no hay nadie",envío,envío,0,G039
178,2026-10-11,whatsapp,"reitero: buenas tardes, necesito cambiar la dirección de entrega dla orden TM-48922, me voy a pasar a Manizales, les agradezco",envío,envío,0,G032
179,2026-10-11,web,"Hola, el producto no corresponde a la foto de la página, me siento engañado, lo devuelvo, gracias",devolución,devolución,0,G016
180,2026-10-11,whatsapp,"Buenas tardes, necesito cambiar la dirección de entrega dla orden TM-48922, me voy a pasar a Manizales, les agradezco",envío,envío,0,G032
181,2026-10-11,whatsapp,"otra vez: hola, el producto no corresponde a la foto de la página, me siento engañado, lo devuelvo, gracias",devolución,devolución,0,G016
182,2026-10-12,correo,"Buenas tardes, me pueden enviar el detalle de lo que pagué? tengo que legalizar el gasto en la oficina, es urgente",facturación,facturación,0,G097
183,2026-10-12,whatsapp,"Parce, cuántos días tengo para devolver el reloj? lo recibí desde hace 4",devolución,devolución,0,G108
184,2026-10-12,whatsapp,"sigo esperando respuesta: buenas tardes, necesito cambiar la dirección de entrega dla orden TM-48922, me voy a pasar a Manizales, les agradezco",envío,envío,0,G032
185,2026-10-12,correo,"de nuevo: qué más, el cargador no carga y los tenis se calienta mucho, prefiero devolverlo antes de que pase algo",devolución,devolución,0,G002
186,2026-10-12,whatsapp,"Qué más, xq me cobraron el domicilio si la promo era envío gratis? pedido TM-49472. mil gracias",facturación,facturación,0,G217
187,2026-10-12,whatsapp,"Qué más, el cargador no carga y los tenis se calienta mucho, prefiero devolverlo antes de que pase algo",devolución,devolución,0,G002
188,2026-10-12,correo,"de nuevo: hola, el producto no corresponde a la foto de la página, me siento engañado, lo devuelvo, gracias",devolución,devolución,0,G016
189,2026-10-13,web,"Buenas, ñapa, me pueden decir si ya salió el pedido? no me llegó ni correo ni nada, un saludo",envío,envío,0,G143
190,2026-10-13,correo,"Hola, la factura no discrimina el IVA y mi contador la necesita así ... quedo atento",facturación,facturación,0,G178
191,2026-10-13,whatsapp,"Buenos días, me cobraron una comisión por pagar contra entrega, nadie me avisó",facturación,facturación,0,G242
192,2026-10-13,correo,"Cordial saludo, compré la camiseta para regalo y no le gustó, aún está sin usar, lo puedo devolver?",devolución,devolución,0,G251
193,2026-10-13,web,por qué me cobraron iva si el producto aparecía sin iva?,facturación,facturación,0,G270
194,2026-10-14,whatsapp,"de nuevo: buenas tardes, la factura no discrimina el IVA y mi contador la necesita asi. gracias",facturación,facturación,0,G028
195,2026-10-14,correo,"Buenas tardes, la factura no discrimina el IVA y mi contador la necesita asi. gracias",facturación,facturación,0,G028
196,2026-10-15,whatsapp,"Cordial saludo, me dijeron que llegaba en 5 días y ya pasó el plazo, qué pasó con la plancha?, gracias",envío,envío,0,G077
197,2026-10-15,web,"Parce, pagué la cuota inicial y me siguen enviando recordatorios de cobro, ya estoy al día",facturación,facturación,0,G175
198,2026-10-16,correo,"buenas, el cupon de bienvenida me lo restaron en un compra pero en el otro no, por que?",facturación,facturación,0,G078
199,2026-10-16,correo,"Buenas, la plancha vino con una falla en la costura, deseo que lo recojan",devolución,devolución,0,G160
200,2026-10-16,whatsapp,"Hola, buenas, me avisaron que el compra TM-47177 salió ayer, pero el link de seguimiento no abre ... mil gracias",envío,envío,0,G284
201,2026-10-17,whatsapp,"pagué con tarjeta y me descontaron, pero el pedido nunca salió. quiero mi plata",facturación,envío,1,G107
202,2026-10-17,whatsapp,"Parce, ya pasaron 4 días hábiles y la orden sigue en 'preparando', se perdió o qué?",envío,envío,0,G280
203,2026-10-18,whatsapp,"Parce, la transportadora dejó el paquete en la portería equivocada, pedido TM-41235",envío,envío,0,G070
204,2026-10-19,correo,"qué más, los tenis huele a usado, parece reempacado, no lo quiero ... quedo atento",devolución,devolución,0,G031
205,2026-10-19,web,"reitero: qué más, los tenis huele a usado, parece reempacado, no lo quiero ... quedo atento",devolución,devolución,0,G031
206,2026-10-20,whatsapp,"Qué más, mi pedido TM-43386 sigue sin llegar, ya van 12 días y el seguimiento no se mueve",envío,envío,0,G102
207,2026-10-20,whatsapp,"me cobraron dos veces y ni siquiera me ha llegado, ya no lo quiero",facturación,devolución,1,G135
208,2026-10-20,correo,me llegó otro color y además la factura dice el color que pedí,devolución,facturación,1,G169
209,2026-10-20,whatsapp,"Parce, quisiera saber el estado del despacho, la página me tiene en 'procesando' hace 5 días",envío,envío,0,G193
210,2026-10-20,web,"Buenas tardes, mi pedido TM-48587 sigue sin llegar, ya van 3 días y la guía no se mueve",envío,envío,0,G236
211,2026-10-20,correo,"pagué envío express y el pedido sigue sin llegar, quiero que me devuelvan ese valor",envío,facturación,1,G250
212,2026-10-20,correo,"que mas, el empaque llego abierto y el morral usado, no lo quiero, devuelvanme la plata",devolución,devolución,0,G261
213,2026-10-20,whatsapp,"Buenas, la factura no discrimina el IVA y mi contador la necesita así",facturación,facturación,0,G265
214,2026-10-21,correo,"Hola, necesito la guia para devolver el termo, ustedes la pagan o yo?",devolución,devolución,0,G047
215,2026-10-21,whatsapp,"el paquete llegó incompleto, me cobraron todo. qué hago?",envío,facturación,1,G274
216,2026-10-22,whatsapp,estoy dentro de los 5 dias de retracto y deseo devolver la orden TM-49568 ... es urgente,devolución,devolución,0,G064
217,2026-10-22,correo,"buenas tardes, el domiciliario no encuentra la dirección, está en Manizales, me pueden llamar?",envío,envío,0,G146
218,2026-10-22,whatsapp,"Parce, dicen que mañana llega y llevo tres mañanas esperando, esto es normal?",envío,envío,0,G154
219,2026-10-22,correo,"la guía dice entregado pero lo recibió otra persona, quiero devolver porque ya no lo necesito",devolución,envío,1,G214
220,2026-10-22,correo,"insisto, buenas, me cobraron el seguro del envio y yo no lo pedi",facturación,facturación,0,G027
221,2026-10-23,whatsapp,"Buenos días, la garantía de el cargador dice un año pero ya me dijeron que no cubre, quiero devolverlo",devolución,devolución,0,G124
222,2026-10-23,web,"Hola, buenas, no deseo el reloj, me arrepentí de la compra. se puede devolver?, es urgente",devolución,devolución,0,G190
223,2026-10-23,whatsapp,"Buenas, me llegó el cobro de la cuota 2 pero yo pagué de contado, revisen porfa ... un saludo",facturación,facturación,0,G197
224,2026-10-23,web,"devolver el producto cuesta lo mismo que el domicilio que me cobraron, qué es eso?",facturación,devolución,1,G231
225,2026-10-23,correo,"necesito cambiar la direccion de entrega del compra TM-48328, me voy a pasar a Bogota",envío,envío,0,G246
226,2026-10-23,whatsapp,"Buenos días, me llegó incompleto: pedí 2 cosas y solo vino 1, y la otra? pedido TM-44264",envío,envío,0,G285
227,2026-10-23,whatsapp,"Buenos dias, el seguimiento figura entregada pero yo no recibi nada, compra TM-42586",envío,envío,0,G294
228,2026-10-24,whatsapp,"Buenas, el mensajero dejó la orden con un desconocido y yo no autoricé eso",envío,envío,0,G050
229,2026-10-24,correo,"no es devolución, es que quiero cambiar la dirección de la factura y del envío",facturación,envío,1,G089
230,2026-10-24,whatsapp,"buenas, me cobraron el seguro del envio y yo no lo pedi",facturación,facturación,0,G027
231,2026-10-25,correo,"Qué más, me descontaron la plata de la tarjeta pero la plataforma dice que el pago fue rechazado (TM-48770) ... gracias",facturación,facturación,0,G052
232,2026-10-25,whatsapp,"otra vez: qué más, pille, se me atrasó el pedido para el cumpleaños, hay forma de acelerarlo?",envío,envío,0,G017
233,2026-10-25,whatsapp,"Cordial saludo, el total del carrito eran $189.900 y me llegó el link de pago por $89.500",facturación,facturación,0,G118
234,2026-10-25,whatsapp,"Hola, la garantía de la plancha dice un año pero ya me dijeron que no cubre, quiero devolverlo",devolución,devolución,0,G151
235,2026-10-25,correo,"Qué más, el cargador no carga y el morral se calienta mucho, prefiero devolverlo antes de que pase algo. es urgente",devolución,devolución,0,G153
236,2026-10-25,web,"Hola, me descontaron la plata de la tarjeta pero la plataforma dice que el pago fue rechazado (TM-46020)",facturación,facturación,0,G203
237,2026-10-25,web,"buenas, me llego dañado la camiseta, adjunto fotos, quiero cambiarlo. es urgente",devolución,devolución,0,G215
238,2026-10-26,web,"Cordial saludo, llegó la caja pero sin la licuadora, solo el empaque, qué pasó en el despacho?",envío,envío,0,G043
239,2026-10-26,correo,"Buenas tardes, ya pagué por transferencia y todavia no me llega el soporte de pago dla orden TM-43958",facturación,facturación,0,G142
240,2026-10-26,correo,"Hola, la factura llegó en dólares y yo pagué en pesos, eso cómo es?",facturación,facturación,0,G222
241,2026-10-26,correo,"Qué más, pille, se me atrasó el pedido para el cumpleaños, hay forma de acelerarlo?",envío,envío,0,G017
242,2026-10-26,whatsapp,"llegó la caja pero sin la camiseta, solo el empaque, qué pasó en el despacho?, es urgente",envío,envío,0,G247
243,2026-10-26,whatsapp,"Hola, buenas, el cargador no carga y la camiseta se calienta mucho, prefiero devolverlo antes de que pase algo",devolución,devolución,0,G268
244,2026-10-26,web,"Buenas tardes, cuántos días tengo para devolver el termo? lo recibí hace 3. quedo atento",devolución,devolución,0,G269
245,2026-10-27,web,"Hola, dicen que mañana llega y llevo tres mañanas esperando, esto es normal? ... un saludo",envío,envío,0,G042
246,2026-10-27,web,"Buenas, envie la orden de vuelta hace 5 dias, cuando me reembolsan?",devolución,devolución,0,G049
247,2026-10-27,web,"Buenas, no entiendo el cargo de $149.500 que aparece en mi cuenta, yo solo compré el reloj",facturación,facturación,0,G075
248,2026-10-27,whatsapp,"cordial saludo, buenas tardes, el domiciliario no encuentra la dirección, está en cali, me pueden llamar?",envío,envío,0,G088
249,2026-10-27,correo,"Buenas, se me pasó el tiempo de retracto por un día, igual me pueden recibir el termo?",devolución,devolución,0,G150
250,2026-10-27,correo,"Qué más, me pueden enviar el detalle de lo que pagué? tengo que legalizar el gasto en la oficina",facturación,facturación,0,G289
251,2026-10-28,whatsapp,"Hola, la licuadora llego defectuoso, no enciende. necesito que me devuelvan la plata",devolución,devolución,0,G048
252,2026-10-28,whatsapp,"Buenos días, me mandaron un producto usado, tiene rayones, quiero cambio por uno nuevo",devolución,devolución,0,G068
253,2026-10-28,whatsapp,"Qué más, ya pasaron 4 días hábiles y el pedido sigue en 'preparando', se perdió o qué?",envío,envío,0,G079
254,2026-10-28,whatsapp,"Buenas, por qué me cobraron IVA si el producto aparecía sin IVA?",facturación,facturación,0,G254
255,2026-10-28,correo,"Cordial saludo, me llego dañado la lampara, adjunto fotos, deseo cambiarlo",devolución,devolución,0,G255
256,2026-10-29,correo,"Buenas tardes, buenas, cuando llega mi compra a Bogota? me urge para el fin de semana",envío,envío,0,G046
257,2026-10-29,whatsapp,"Buenas, no me llegó el código de seguimiento, de qué forma hago para rastrear la lámpara?",envío,envío,0,G101
258,2026-10-29,whatsapp,"Hola, la factura trae mal la cédula, favor corregirla, pedido TM-40055",facturación,facturación,0,G187
259,2026-10-29,correo,"insisto, hola, buenas, me pueden enviar el detalle de lo que pagué? tengo que legalizar el gasto en la oficina. mil gracias",facturación,facturación,0,G007
260,2026-10-29,whatsapp,"buenos días, solicito copia de la factura de el reloj para la garantía del fabricante, un saludo",facturación,facturación,0,G266
261,2026-10-29,correo,"me llegó dañado el termo y además me cobraron el envío, quiero la plata de vuelta",devolución,facturación,1,G271
262,2026-10-29,correo,"Buenas, el pedido salió para Envigado pero yo vivo en otro barrio, se va a devolver a bodega?",envío,envío,0,G276
263,2026-10-29,web,"en la página decía $89.500 y en el extracto me cobraron $59.900, a qué se debe la diferencia? ... un saludo",facturación,facturación,0,G286
264,2026-10-30,correo,"Buenas, quiero devolver el reloj, no es lo que esperaba. pedido TM-48017",devolución,devolución,0,G069
265,2026-10-30,whatsapp,"reitero: hola, pagué con PSE y el banco dice aprobado pero ustedes no registran nada, compra TM-45259",facturación,facturación,0,G008
266,2026-10-30,whatsapp,"Buenas tardes, pille, me aparece un cargo pendiente de $79.000 y no se de que es",facturación,facturación,0,G132
267,2026-10-30,whatsapp,"de nuevo: hola, pagué con PSE y el banco dice aprobado pero ustedes no registran nada, compra TM-45259",facturación,facturación,0,G008
268,2026-10-30,whatsapp,"la guía figura entregada pero yo no recibí nada, pedido TM-44716",envío,envío,0,G174
269,2026-10-30,web,"Hola, buenas, me pueden enviar el detalle de lo que pagué? tengo que legalizar el gasto en la oficina. mil gracias",facturación,facturación,0,G007
270,2026-10-30,whatsapp,"sigo esperando respuesta: hola, pagué con PSE y el banco dice aprobado pero ustedes no registran nada, compra TM-45259",facturación,facturación,0,G008
271,2026-10-30,whatsapp,"necesito la factura para poder hacer la devolución, no me la han enviado",facturación,devolución,1,G240
272,2026-10-30,whatsapp,el descuento no se aplicó y por eso quiero devolver todo,facturación,devolución,1,G260
273,2026-10-31,web,necesito que corrijan la factura para poder reclamar la garantía del termo,facturación,facturación,1,G204
274,2026-10-31,web,"Hola, pagué con PSE y el banco dice aprobado pero ustedes no registran nada, compra TM-45259",facturación,facturación,0,G008
275,2026-10-31,correo,"Cordial saludo, el total del carrito eran $59.500 y me llegó el link de pago por $89.500",facturación,facturación,0,G298
276,2026-11-01,whatsapp,"buenas, la talla no me sirve, como hago el cambio? compre los audifonos desde hace 6 dias",devolución,devolución,0,G058
277,2026-11-01,web,"Buenos dias, el mensajero dice que fue a mi casa y no habia nadie, pero yo estuve todo el dia",envío,envío,0,G084
278,2026-11-01,correo,"otra vez: buenas tardes, llevo 12 días esperando el reloj y el seguimiento no avanza desde Pereira, es urgente",envío,envío,0,G030
279,2026-11-01,whatsapp,"Buenas tardes, llevo 12 días esperando el reloj y el seguimiento no avanza desde Pereira, es urgente",envío,envío,0,G030
280,2026-11-01,whatsapp,"buenos días, solicito cambio por garantía de el reloj, pedido tm-47811, compré en rionegro",devolución,devolución,0,G218
281,2026-11-01,whatsapp,"de nuevo: buenas tardes, llevo 12 días esperando el reloj y el seguimiento no avanza desde Pereira, es urgente",envío,envío,0,G030
282,2026-11-01,whatsapp,"Buenas tardes, la talla no me sirve, de qué forma hago el cambio? compré la chaqueta hace 4 días. un saludo",devolución,devolución,0,G257
283,2026-11-01,correo,"Buenos dias, en que horario pasan a entregar en Bello? trabajo hasta las 6",envío,envío,0,G279
284,2026-11-02,correo,"buenas tardes, me avisaron que el pedido tm-45175 salió ayer, pero el link de seguimiento no abre",envío,envío,0,G061
285,2026-11-03,whatsapp,"Hola, buenas, dicen que mañana llega y llevo tres mañanas esperando, esto es normal?",envío,envío,0,G073
286,2026-11-03,web,la transportadora me cobró contra entrega algo que ya había pagado en línea,facturación,envío,1,G290
287,2026-11-04,whatsapp,"Hola, ñapa, me pueden decir si ya salió el compra? no me llegó ni correo ni nada",envío,envío,0,G044
288,2026-11-04,whatsapp,"Cordial saludo, el paquete está detenido en una bodega de Barranquilla hace 6 días, pueden escalarlo?",envío,envío,0,G059
289,2026-11-04,web,"Buenos días, necesito la factura electrónica del compra TM-41637, no me llegó al correo, mil gracias",facturación,facturación,0,G066
290,2026-11-04,web,"Qué más, quiero el certificado de retención del año, somos clientes frecuentes",facturación,facturación,0,G087
291,2026-11-04,whatsapp,"Hola, buenas, devolví los tenis y me dijeron que llegó en mal estado, pero salió perfecto de mi casa",devolución,devolución,0,G099
292,2026-11-04,correo,"Cordial saludo, el mensajero dice que fue a mi casa y no había nadie, pero yo estuve todo el día. mil gracias",envío,envío,0,G167
293,2026-11-04,whatsapp,"Hola, necesito que entreguen en horario de oficina, en la tarde no hay nadie",envío,envío,0,G206
294,2026-11-05,whatsapp,"Buenas tardes, en qué horario pasan a entregar en Medellín? trabajo hasta las 6. mil gracias",envío,envío,0,G123
295,2026-11-05,correo,"Hola, me mandaron la chaqueta equivocado, yo pedí otra referencia, quiero devolverlo",devolución,devolución,0,G201
296,2026-11-06,whatsapp,"Buenas, el paquete está detenido en una bodega de Rionegro desde hace 6 días, pueden escalarlo?. les agradezco",envío,envío,0,G086
297,2026-11-06,correo,quiero cancelar el pedido porque se demora mucho y que me devuelvan lo pagado,envío,devolución,1,G161
298,2026-11-06,correo,"otra vez: hola, pille, se me atrasó el compra para el cumpleaños, hay forma de acelerarlo? ... un saludo",envío,envío,0,G011
299,2026-11-06,whatsapp,"Qué más, quisiera saber el estado del despacho, la página me tiene en 'procesando' desde hace 10 días",envío,envío,0,G273
300,2026-11-07,web,"Hola, buenas, la transportadora dejo el paquete en la porteria equivocada, pedido TM-48609",envío,envío,0,G041
301,2026-11-07,whatsapp,"Que mas, requiero una nota credito dla orden TM-43899, contabilidad me la esta pidiendo",facturación,facturación,0,G051
302,2026-11-07,whatsapp,"Buenas tardes, xq me cobraron el domicilio si la promo era envío gratis? pedido TM-43153",facturación,facturación,0,G094
303,2026-11-07,whatsapp,"cordial saludo, pille, me aparece un cargo pendiente de $189.500 y no sé de qué es",facturación,facturación,0,G121
304,2026-11-07,correo,"Buenos días, ya pagué por transferencia y todavia no me llega el soporte de pago del pedido TM-49368",facturación,facturación,0,G147
305,2026-11-08,whatsapp,"Parce, el mensajero dejó el pedido con un desconocido y yo no autoricé eso",envío,envío,0,G065
306,2026-11-08,correo,devolví el pedido hace dos semanas y todavía no me reversan el pago,devolución,facturación,1,G090
307,2026-11-08,correo,"Hola, pille, se me atrasó el compra para el cumpleaños, hay forma de acelerarlo? ... un saludo",envío,envío,0,G011
308,2026-11-08,whatsapp,"Buenas tardes, la lámpara llegó defectuoso, no enciende. necesito que me devuelvan la plata, mil gracias",devolución,devolución,0,G205
309,2026-11-08,web,"Parce, el mensajero dice que fue a mi casa y no habia nadie, pero yo estuve todo el dia",envío,envío,0,G208
310,2026-11-08,whatsapp,"Que mas, me mandaron el reloj equivocado, yo pedi otra referencia, quiero devolverlo. un saludo",devolución,devolución,0,G216
311,2026-11-08,whatsapp,"no me llegó el código de seguimiento, cómo hago para rastrear la camiseta? ... gracias",envío,envío,0,G235
312,2026-11-09,whatsapp,"insisto, buenos días, el total del carrito eran $39.500 y me llegó el link de pago por $79.900 ... un saludo",facturación,facturación,0,G033
313,2026-11-09,whatsapp,"Qué más, estoy dentro de los 5 días de retracto y quiero devolver el pedido TM-44901",devolución,devolución,0,G056
314,2026-11-09,whatsapp,"hola, no me llegó el código de seguimiento, cómo hago para rastrear la plancha?",envío,envío,0,G076
315,2026-11-09,web,"me llego incompleto: pedi 2 cosas y solo vino 1, y la otra? pedido TM-44770",envío,envío,0,G159
316,2026-11-10,correo,"Buenos días, el total del carrito eran $39.500 y me llegó el link de pago por $79.900 ... un saludo",facturación,facturación,0,G033
317,2026-11-10,whatsapp,"Buenas, necesito cambiar la dirección de entrega del pedido TM-40840, me voy a pasar a Pereira. les agradezco",envío,envío,0,G164
318,2026-11-10,whatsapp,"otra vez: buenas, deseo que me recojan el termo en la casa, no puedo ir a ninguna oficina",devolución,devolución,0,G013
319,2026-11-11,whatsapp,"hola, buenas, necesito cambiar la dirección de entrega del pedido tm-41700, me voy a pasar a itagüí. es urgente",envío,envío,0,G037
320,2026-11-11,whatsapp,"Buenos días, el empaque llegó abierto y el termo usado, no lo quiero, devuélvanme la plata",devolución,devolución,0,G113
321,2026-11-11,whatsapp,"Buenas, compré la camiseta para regalo y no le gustó, aún está sin usar, lo puedo devolver?",devolución,devolución,0,G183
322,2026-11-11,whatsapp,"Buenas, deseo que me recojan el termo en la casa, no puedo ir a ninguna oficina",devolución,devolución,0,G013
323,2026-11-12,whatsapp,"Hola, buenas, me cobraron el seguro del envío y yo no lo pedí",facturación,facturación,0,G127
324,2026-11-13,whatsapp,"Parce, la transportadora dejó el paquete en la portería equivocada, pedido TM-48827",envío,envío,0,G022
325,2026-11-13,correo,"mi pedido llegó roto, pero lo que más me molesta es que lo cobraron por adelantado",devolución,facturación,1,G085
326,2026-11-13,correo,"Parce, me cobraron el seguro del envio y yo no lo pedi",facturación,facturación,0,G144
327,2026-11-13,correo,"de nuevo: parce, la transportadora dejó el paquete en la portería equivocada, pedido TM-48827",envío,envío,0,G022
328,2026-11-13,correo,"Qué más, de qué forma es el proceso de garantía? la lámpara dejó de funcionar a las 2 semanas",devolución,devolución,0,G170
329,2026-11-13,web,"redeseo que entreguen en horario de oficina, en la tarde no hay nadie",envío,envío,0,G212
330,2026-11-13,whatsapp,"reitero: parce, la transportadora dejó el paquete en la portería equivocada, pedido TM-48827",envío,envío,0,G022
331,2026-11-14,correo,"insisto, hola, ñapa, me pueden decir si ya salió la orden? no me llegó ni correo ni nada",envío,envío,0,G024
332,2026-11-14,web,"el morral vino con una pieza rota y sin manual, lo quiero devolver",devolución,devolución,0,G053
333,2026-11-14,correo,"ya no quiero el pedido, aún no sale, me devuelven la plata?",devolución,envío,1,G229
334,2026-11-14,whatsapp,"sigo esperando respuesta: parce, la transportadora dejó el paquete en la portería equivocada, pedido TM-48827",envío,envío,0,G022
335,2026-11-14,whatsapp,"Hola, ñapa, me pueden decir si ya salió la orden? no me llegó ni correo ni nada",envío,envío,0,G024
336,2026-11-14,whatsapp,"cordial saludo, me llegó el cobro de la cuota 2 pero yo pagué de contado, revisen por les pido el favor de",facturación,facturación,0,G282
337,2026-11-15,correo,"hola, buenas, me equivoqué de talla en la orden tm-40639, hay cambio gratis?",devolución,devolución,0,G045
338,2026-11-15,whatsapp,"Que mas, la plancha no calienta parejo, me parece un defecto de fabrica, deseo cambio ... gracias",devolución,devolución,0,G120
339,2026-11-15,whatsapp,"Hola, buenas, me mandaron un producto usado, tiene rayones, deseo cambio por uno nuevo",devolución,devolución,0,G165
340,2026-11-15,web,"Parce, no deseo devolver nada, solo saber dónde está mi pedido TM-45737",envío,envío,0,G238
341,2026-11-16,whatsapp,"Buenas, quiero que me recojan la plancha en la casa, no puedo ir a ninguna oficina",devolución,devolución,0,G067
342,2026-11-16,correo,"Buenas, el descuento del cupón no se aplicó al pedido TM-49105, pagué de más",facturación,facturación,0,G093
343,2026-11-16,whatsapp,"Cordial saludo, ya no necesito los tenis, lo compré doble por error, cómo lo devuelvo?",devolución,devolución,0,G141
344,2026-11-17,whatsapp,"necesito que le cambien el nombre a la factura, salió con el de mi mamá",facturación,facturación,0,G228
'''
open('mensajes_servicio.csv', 'w', encoding='utf-8').write(CSV_TEXTO)
print('Archivo guardado:', len(CSV_TEXTO.splitlines()) - 1, 'filas')

In [ ]:
df = pd.read_csv("mensajes_servicio.csv")

# La misma partición por grupo de la clase 4 (misma semilla, mismo código)
g1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEMILLA)
idx_resto, idx_prueba = next(g1.split(df, groups=df["grupo"]))
resto = df.iloc[idx_resto]
g2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEMILLA)
i_ent, i_val = next(g2.split(resto, groups=resto["grupo"]))
df["conjunto"] = "entrenamiento"
df.loc[resto.index[i_val], "conjunto"] = "validación"
df.loc[df.index[idx_prueba], "conjunto"] = "prueba"

print(df["conjunto"].value_counts().to_dict())
print("La prueba se queda sin abrir:", int((df["conjunto"] == "prueba").sum()), "mensajes.")

ent = df[df["conjunto"] == "entrenamiento"].copy()
va = df[df["conjunto"] == "validación"].copy()

**Comprueba:** deben salir 208 de entrenamiento, 66 de validación y 70 de prueba, igual que en la clase 4. Si no, algo cambió en el archivo o en la semilla.

La referencia de palabras clave de la clase 4 vuelve a aparecer porque es contra lo que se compara todo.

In [ ]:
CLASE_MAYOR = ent["etiqueta"].mode()[0]
PALABRAS = {
    "facturación": ["factur", "cobr", "iva", "nit", "pago", "pagu", "nota crédito", "nota credito", "valor", "cargo", "cuota", "descuento", "cupón", "cupon", "extracto"],
    "envío":       ["guía", "guia", "llega", "llegó", "llego", "entrega", "dirección", "direccion", "mensajero", "paquete", "despacho", "seguimiento", "transportadora", "domicilio", "pedido"],
    "devolución":  ["devol", "reembols", "cambio", "garantía", "garantia", "defectuos", "dañado", "danado", "talla", "retracto", "no me sirve", "no lo quiero"],
}
def por_palabras(mensaje):
    m = mensaje.lower()
    puntos = {c: sum(p in m for p in lista) for c, lista in PALABRAS.items()}
    mejor = max(puntos.values())
    if mejor == 0:
        return CLASE_MAYOR
    return [c for c in CLASES if puntos[c] == mejor][0]

def resumen(reales, predichas, nombre):
    aciertos = int((np.array(reales) == np.array(predichas)).sum())
    return {"método": nombre, "aciertos": f"{aciertos} de {len(reales)}",
            "exactitud": round(accuracy_score(reales, predichas), 2),
            "F1 macro": round(f1_score(reales, predichas, labels=CLASES, average="macro", zero_division=0), 2)}

va["pred_palabras"] = va["mensaje"].apply(por_palabras)
print(resumen(va["etiqueta"], va["pred_palabras"], "palabras clave"))

## Parte 2 · Tres caminos para usar un modelo que ya existe
Un modelo preentrenado ya aprendió algo del idioma. La pregunta de hoy es **cuánto de él tocar**:

| Camino | Qué cambia | Qué necesita | Cuándo se justifica | Riesgo principal |
|---|---|---|---|---|
| **A · Tal cual** | Nada. Solo cambias cómo lo usas: las etiquetas, la plantilla o el prompt | Ninguna etiqueta propia | Tus clases se entienden por su significado y hay pocos datos | No se adapta a tu jerga |
| **B · Salida entrenada sobre representaciones congeladas** | Solo una capa pequeña que decide la clase. El modelo grande queda **congelado** | Cientos de ejemplos etiquetados | El modelo entiende el idioma pero no tus clases | Sobreajuste con pocos datos o duplicados |
| **C · Afinar parte del modelo** | Se actualizan las últimas capas (millones de parámetros) | Miles de ejemplos, GPU y tiempo | B no alcanza y hay datos de sobra | Costo, olvidar lo aprendido, difícil de revertir |

**Hoy ejecutamos A y B. C solo se diseña:** no la entrenamos y no la reportamos como hecha.

**Regla del experimento: un cambio a la vez.** Si cambias las etiquetas y la plantilla juntas, no sabes cuál movió el resultado. Todo lo que probemos se anota en el cuaderno `LOG` con su único cambio.

## Parte 3 · Camino A: usar el modelo tal cual
Es el mismo modelo de la clase 4 (`Recognai/bert-base-spanish-wwm-cased-xnli`: BETO, español, entrenado en inferencia de lenguaje natural, 3 etiquetas entailment/neutral/contradiction, unos 440 MB).

Si guardaste `predicciones_base.csv` en la clase 4 y lo subiste a Colab, lo usamos. Si no, el notebook calcula las 66 predicciones de validación (unos minutos en CPU).

In [ ]:
PLANTILLA = "Este ejemplo es {}."
MODELO_NLI = "Recognai/bert-base-spanish-wwm-cased-xnli"

if os.path.exists("predicciones_base.csv"):
    base = pd.read_csv("predicciones_base.csv")[["id", "pred_modelo", "confianza"]]
    va = va.merge(base, on="id", how="left")
    print("Usé predicciones_base.csv de la clase 4.")
else:
    clf = pipeline("zero-shot-classification", model=MODELO_NLI, device=0 if torch.cuda.is_available() else -1)
    t0 = time.time()
    res = clf(va["mensaje"].tolist(), candidate_labels=CLASES, hypothesis_template=PLANTILLA, batch_size=16)
    va["pred_modelo"] = [r["labels"][0] for r in res]
    va["confianza"] = [round(r["scores"][0], 3) for r in res]
    print(f"Calculé {len(va)} predicciones en {time.time() - t0:.0f} s.")

registrar("A0 modelo tal cual", "punto de partida (plantilla «Este ejemplo es {}.», etiquetas originales)", va["etiqueta"], va["pred_modelo"])
registrar("referencia: palabras clave", "no es un modelo, solo la regla a mano", va["etiqueta"], va["pred_palabras"])
pd.DataFrame(LOG)

**Un cambio, solo uno:** cambiamos **únicamente el nombre de las etiquetas** y dejamos todo lo demás igual (modelo, plantilla, mensajes). Antes de ejecutar, escribe si crees que mejorará, empeorará o dará igual.

In [ ]:
ETIQ_ALT = {"facturación": "cobros y facturas", "envío": "entrega del pedido", "devolución": "devolver o cambiar el producto"}
inversa = {v: k for k, v in ETIQ_ALT.items()}

if "clf" not in globals():
    clf = pipeline("zero-shot-classification", model=MODELO_NLI, device=0 if torch.cuda.is_available() else -1)
res_alt = clf(va["mensaje"].tolist(), candidate_labels=list(ETIQ_ALT.values()), hypothesis_template=PLANTILLA, batch_size=16)
va["pred_modelo_alt"] = [inversa[r["labels"][0]] for r in res_alt]

registrar("A1 etiquetas con otras palabras", "solo cambia el nombre de las etiquetas", va["etiqueta"], va["pred_modelo_alt"])
cambiaron = int((va["pred_modelo"] != va["pred_modelo_alt"]).sum())
print("Mensajes cuya predicción cambió:", cambiaron, "de", len(va))
pd.DataFrame(LOG)

**Cómo leerlo:** fíjate en cuántos mensajes **cambiaron de predicción**, no solo en cuántos aciertos hay. Pueden cambiar 8 mensajes y que el acierto quede igual porque unos mejoraron y otros empeoraron. Con 66 mensajes, dos o tres aciertos de diferencia pueden ser azar.

## Parte 4 · Camino A con ejemplos en el prompt (chat de IA generativa)
Otra forma de «no tocar el modelo»: darle ejemplos dentro del prompt. El modelo del chat **no cambia sus parámetros**: solo lee tus ejemplos como parte de su contexto. Eso consume tokens y cabe solo hasta cierto tamaño.

Armamos el prompt con 3 ejemplos de cada clase, **sacados solo del entrenamiento**, y el mismo lote de 30 mensajes de validación de la clase 4.

In [ ]:
ejemplos = (ent[ent["ambiguo"] == 0].groupby("etiqueta").sample(3, random_state=SEMILLA)
            .sample(frac=1, random_state=SEMILLA))
bloque_ej = "\n".join(f"Mensaje: {m}\nEtiqueta: {e}\n" for m, e in zip(ejemplos["mensaje"], ejemplos["etiqueta"]))

PROMPT = f"""Clasifica cada mensaje de servicio al cliente de una tienda en línea ficticia.
Etiquetas posibles: facturación, envío, devolución.
Responde solo con una línea por mensaje, con el formato "número: etiqueta". Sin explicaciones.

Ejemplos:

{bloque_ej}
Mensajes por clasificar:
"""

lote = va.sample(30, random_state=SEMILLA).reset_index(drop=True)
texto_lote = "\n".join(f"{i + 1}. {m}" for i, m in enumerate(lote["mensaje"]))
print(PROMPT + texto_lote)
print("\n--- Tokens aproximados del prompt:", len((PROMPT + texto_lote).split()) * 2, "---")

In [ ]:
# Pega entre las comillas la respuesta de tu chat (una línea por mensaje, por ejemplo "1: envío").
RESPUESTA_CHAT = """

"""

def leer_respuesta(texto, n):
    quitar = str.maketrans("áéíóú", "aeiou")
    salida = ["sin clasificar"] * n
    for num, palabra in re.findall(r"(\d+)\s*[:.\-)]\s*([A-Za-zÁÉÍÓÚáéíóúñ]+)", texto):
        i = int(num) - 1
        for c in CLASES:
            if palabra.lower().translate(quitar) == c.translate(quitar) and 0 <= i < n:
                salida[i] = c
    return salida

if RESPUESTA_CHAT.strip():
    lote["pred_chat"] = leer_respuesta(RESPUESTA_CHAT, len(lote))
    lote["pred_palabras"] = lote["mensaje"].apply(por_palabras)
    lote["pred_modelo"] = va.set_index("id").loc[lote["id"], "pred_modelo"].values
    print("Etiquetas no reconocidas:", int((lote["pred_chat"] == "sin clasificar").sum()))
    registrar("A2 chat con 9 ejemplos en el prompt (lote de 30)", "se agregan ejemplos al prompt", lote["etiqueta"], lote["pred_chat"])
    print(pd.DataFrame([resumen(lote["etiqueta"], lote["pred_palabras"], "palabras clave"),
                        resumen(lote["etiqueta"], lote["pred_modelo"], "modelo preentrenado (zero-shot)"),
                        resumen(lote["etiqueta"], lote["pred_chat"], "chat con ejemplos")]))
else:
    print("Pega la respuesta de tu chat arriba y vuelve a ejecutar esta celda.")

**Comparación justa:** si en la clase 4 hiciste este mismo lote **sin ejemplos**, ya tienes la medición para saber si los ejemplos ayudaron. Un chat que cambia de una vez a otra necesita repetirse; anota el nombre y la versión del chat.

## Parte 5 · Camino B: representaciones congeladas y una salida entrenada por ti
### Qué es una representación
Un modelo de este tipo (un *encoder*) convierte una frase en una lista de números (un **vector**). Frases con el mismo significado quedan con vectores parecidos. Ese vector es la **representación**. Si el modelo grande queda **congelado** (no se actualiza ningún parámetro), podemos entrenar encima una salida muy pequeña que mire esos números y decida la clase.

### El modelo que usamos para esto
| Pregunta | Respuesta |
|---|---|
| Nombre | `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` |
| ¿Qué es? | Un encoder de la familia BERT (12 capas), de 118 millones de parámetros, que devuelve 384 números por frase |
| ¿Idioma? | 50 idiomas, entre ellos español |
| ¿Para qué se entrenó? | Para que frases con el mismo significado queden cerca entre sí (similitud de oraciones) |
| ¿Qué NO tiene? | Ninguna salida de clases: no sabe qué es «facturación» ni «envío» |
| Licencia y tamaño | Apache 2.0, unos 470 MB. Lee hasta 128 tokens |

Un modelo distinto al de la clase 4: aquel decide si una hipótesis se sigue del mensaje; este solo produce vectores.

In [ ]:
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tok_emb = AutoTokenizer.from_pretrained(MODELO_EMB)
enc = AutoModel.from_pretrained(MODELO_EMB)
enc.eval()
for p in enc.parameters():
    p.requires_grad = False          # congelado: ningún parámetro del modelo se va a actualizar

total = sum(p.numel() for p in enc.parameters())
entrenables = sum(p.numel() for p in enc.parameters() if p.requires_grad)
print(f"Parámetros del modelo: {total / 1e6:.0f} millones | entrenables ahora: {entrenables}")

**De texto a vector.** El modelo da un vector por cada token; promediamos los de la frase (ignorando el relleno) y queda un solo vector por mensaje. Es lo que recomienda la ficha del modelo.

In [ ]:
def incrustar(textos, lote=32):
    """Devuelve una matriz (mensajes x 384) con el vector de cada mensaje."""
    salidas = []
    for i in range(0, len(textos), lote):
        ent_ = tok_emb(textos[i:i + lote], padding=True, truncation=True, max_length=128, return_tensors="pt")
        with torch.no_grad():
            h = enc(**ent_).last_hidden_state                 # (frases, tokens, 384)
        m = ent_["attention_mask"].unsqueeze(-1).float()      # 1 en tokens reales, 0 en relleno
        salidas.append(((h * m).sum(1) / m.sum(1)).numpy())   # promedio sobre los tokens reales
    return np.vstack(salidas)

t0 = time.time()
X = incrustar(df["mensaje"].tolist())
print(f"Matriz de representaciones: {X.shape} en {time.time() - t0:.0f} s")
print("Primeros 5 números del primer mensaje:", np.round(X[0, :5], 3))

**¿Se parecen de verdad los vectores de frases con el mismo significado?** Comparamos con la **similitud coseno** (1 = misma dirección, 0 = nada que ver). Escribe antes qué esperas.

In [ ]:
def coseno(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

par = incrustar(["me cobraron dos veces el pedido", "me hicieron un cargo doble por la compra", "mi paquete no ha llegado todavía"])
print("cobro doble  vs  cargo doble :", round(coseno(par[0], par[1]), 2), "(mismo significado, otras palabras)")
print("cobro doble  vs  paquete     :", round(coseno(par[0], par[2]), 2), "(otro tema)")

### Entrenar la salida
Entrenamos una **regresión logística**: una capa que multiplica el vector de 384 números por una matriz de pesos y da una probabilidad por clase. Se entrena **solo con los 208 mensajes de entrenamiento** y se mide en validación. Son 384 × 3 + 3 = 1.155 números por aprender, contra 118 millones que quedan intactos.

In [ ]:
y = df["etiqueta"].values
m_ent = (df["conjunto"] == "entrenamiento").values
m_val = (df["conjunto"] == "validación").values

cabeza = LogisticRegression(max_iter=2000, C=1.0)
cabeza.fit(X[m_ent], y[m_ent])
va["pred_B1"] = cabeza.predict(X[m_val])

print("Números que se entrenaron:", cabeza.coef_.size + cabeza.intercept_.size, "| los del modelo grande, congelados:", f"{total / 1e6:.0f} millones")
registrar("B1 salida logística sobre representaciones congeladas", "se entrena una salida con los 208 mensajes de entrenamiento (C=1)", va["etiqueta"], va["pred_B1"])
print(classification_report(va["etiqueta"], va["pred_B1"], labels=CLASES, zero_division=0))
pd.DataFrame(LOG)

**¿Cuánto se puede creer este número?** Con 66 mensajes, poco. Dos revisiones para calibrar:

1. **Cuántos datos hacen falta.** Entrenamos con el 25 %, el 50 % y el 100 % del entrenamiento, cinco veces cada una con una muestra distinta. Si con la mitad se obtiene casi lo mismo, conseguir más etiquetas no es lo urgente.
2. **Cuánto varía por azar.** Mira el rango (mínimo y máximo) de cada fila: eso es lo que no puedes distinguir de una mejora real.

In [ ]:
rng = np.random.RandomState(SEMILLA)
idx_ent = np.where(m_ent)[0]
filas = []
for frac in [0.25, 0.5, 1.0]:
    ks = []
    for rep in range(5):
        n = int(frac * len(idx_ent))
        sel = idx_ent if frac == 1.0 else rng.choice(idx_ent, size=n, replace=False)
        mod_ = LogisticRegression(max_iter=2000).fit(X[sel], y[sel])
        ks.append(int((mod_.predict(X[m_val]) == y[m_val]).sum()))
    filas.append({"datos de entrenamiento": f"{int(frac * 100)} % ({int(frac * len(idx_ent))} mensajes)",
                  "aciertos (mín - máx)": f"{min(ks)} - {max(ks)} de {int(m_val.sum())}"})
pd.DataFrame(filas)

**Un cambio más, solo uno:** la fuerza de regularización `C` (un valor bajo hace que el modelo se aferre menos a los datos de entrenamiento). Todo lo demás, igual que B1.

In [ ]:
for C_ in [0.1, 10.0]:
    mod_ = LogisticRegression(max_iter=2000, C=C_).fit(X[m_ent], y[m_ent])
    p = mod_.predict(X[m_val])
    registrar(f"B2 salida logística con C={C_}", f"solo cambia C (de 1 a {C_})", va["etiqueta"], p)
pd.DataFrame(LOG)

## Parte 6 · Camino C: afinar parte del modelo (aquí solo se diseña)
Afinar es entrenar también algunas capas del modelo grande, además de la salida. **No lo entrenamos hoy.** Lo que sí hacemos es medir lo que costaría, para decidir con números y no por intuición.

In [ ]:
def parametros(*prefijos):
    return sum(p.numel() for n, p in enc.named_parameters() if any(n.startswith(x) for x in prefijos))

salida_p = 384 * 3 + 3
ultimas2 = parametros("encoder.layer.10.", "encoder.layer.11.")
filas = []
for nombre, entrenables_ in [("B · solo la salida", salida_p),
                             ("C1 · últimas 2 capas + salida", ultimas2 + salida_p),
                             ("C2 · todo el modelo + salida", total + salida_p)]:
    # Estimación en float32: pesos (4 B) + por cada parámetro entrenable, gradiente (4 B) y dos estados de Adam (8 B)
    mb = (total * 4 + entrenables_ * 12) / 1e6
    filas.append({"camino": nombre, "parámetros que se actualizan": f"{entrenables_:,}".replace(",", "."),
                  "% del modelo": f"{100 * entrenables_ / total:.2f} %", "memoria aprox. (MB)": round(mb)})
pd.DataFrame(filas)

**Cómo leerlo:** la memoria es una estimación sin contar las activaciones, que crecen con el largo del texto y el tamaño del lote. Sirve para ver el orden de magnitud: afinar pasa de miles de números a millones, y de segundos a minutos u horas según el hardware.

**Qué se necesitaría para justificar C:** una hipótesis concreta (qué error esperas que baje), suficientes ejemplos etiquetados por clase, una forma de no mirar la prueba mientras ajustas y un plan para volver al punto anterior si empeora. Si no lo tienes, B alcanza para esta clase.

> Regla de honestidad: si no entrenaste algo, no lo cuentes como hecho. En tu ficha, el camino C queda como *diseño*, no como resultado.

## Parte 7 · Comparar y mirar qué errores cambian
Un número de aciertos no dice **qué mensajes** cambiaron. Comparamos el modelo tal cual (A0) con la salida entrenada (B1) mensaje por mensaje.

In [ ]:
tabla = pd.DataFrame([
    resumen(va["etiqueta"], va["pred_palabras"], "palabras clave"),
    resumen(va["etiqueta"], va["pred_modelo"], "A0 · modelo tal cual"),
    resumen(va["etiqueta"], va["pred_modelo_alt"], "A1 · otras palabras en las etiquetas"),
    resumen(va["etiqueta"], va["pred_B1"], "B1 · salida entrenada (representaciones congeladas)"),
])
print(tabla.to_string(index=False))

err_A = set(va.loc[va["pred_modelo"] != va["etiqueta"], "id"])
err_B = set(va.loc[va["pred_B1"] != va["etiqueta"], "id"])
print()
print("Errores de A0:", len(err_A), "| de B1:", len(err_B))
print("Los arregló B1:", len(err_A - err_B), "| los dañó B1 (errores nuevos):", len(err_B - err_A), "| persisten:", len(err_A & err_B))

In [ ]:
persisten = va[va["id"].isin(err_A & err_B)]
persisten[["id", "mensaje", "etiqueta", "etiqueta_revisor2", "ambiguo", "pred_modelo", "pred_B1"]]

**Cómo leerlo:**
- Los errores que **persisten** en A0 y B1 son los candidatos a *etiqueta* o *alcance*: si un segundo revisor habría puesto otra clase (`etiqueta_revisor2`) o el mensaje es ambiguo, cambiar el modelo no ayuda.
- Los errores que **dañó B1** son un aviso: una salida entrenada con pocos ejemplos puede aprender atajos del entrenamiento.
- Si B1 arregla pocos y daña casi los mismos, la diferencia de aciertos no sostiene una decisión.

Piensa en tus dos errores de la clase 4: ¿siguen estando? ¿qué camino los arreglaría?

## Parte 8 · Tu ficha de hipótesis
Una hipótesis dice **qué cambio, qué esperas que pase y cómo lo vas a medir**, antes de ejecutar. Completa los campos. Si dejas alguno vacío, la celda te avisa cuál falta.

Reglas de la ficha:
- **Un solo cambio** respecto a la medición base.
- Todo lo demás **queda igual**: partición, semilla, mensajes de validación.
- El **criterio de decisión se escribe antes** de ver el resultado (por ejemplo, cuántos aciertos de diferencia y en cuántas repeticiones).
- La **prueba** no se abre hasta la comparación final.
- Lo que no entrenaste, **no se cuenta como hecho**.

In [ ]:
HIPOTESIS = {
    "camino": "",                       # A, B o C
    "hipotesis": "",                    # "Si [cambio], entonces [qué error baja] porque [razón]"
    "un_solo_cambio": "",               # exactamente una cosa
    "lo_que_queda_igual": ["partición por grupo", "semilla 1540", "66 mensajes de validación"],
    "error_que_se_espera_arreglar": "", # id y tipo de error (modelo, etiqueta o alcance)
    "medida": "",                       # por ejemplo, aciertos en validación y cuáles de los dos errores cambian
    "criterio_de_decision": "",         # escrito ANTES de ejecutar
    "riesgos": "",                      # por ejemplo, sobreajuste, casi-duplicados, pocos datos
    "lo_que_no_afirmo": "",             # lo que aún no puedes decir con estos datos
    "prueba": "sin abrir hasta la comparación final",
}

faltan = [k for k, v in HIPOTESIS.items() if v in ("", [])]
print("Campos por completar:", faltan if faltan else "ninguno")

ficha = {
    "datos": {"archivo": "mensajes_servicio.csv", "semilla": SEMILLA, "partición": df["conjunto"].value_counts().to_dict()},
    "hipotesis": HIPOTESIS,
    "experimentos": LOG,
}
with open("hipotesis_experimento.json", "w", encoding="utf-8") as f:
    json.dump(ficha, f, ensure_ascii=False, indent=2)
pd.DataFrame(LOG).to_csv("registro_experimentos.csv", index=False)
print(json.dumps(ficha["hipotesis"], ensure_ascii=False, indent=2))

Descarga `hipotesis_experimento.json` y `registro_experimentos.csv` (panel de archivos de Colab, a la izquierda): son tu ficha y tu cuaderno de experimentos. En la clase 6 partimos de ellos.

**Preguntas para cerrar:**
1. ¿Qué camino habrías escogido sin medir, y cambió después de ver los números?
2. ¿Qué cambio concreto probarías primero y qué resultado te haría abandonarlo?
3. ¿Qué parte de tu ficha sigue siendo una suposición y no un resultado?